# 数理変換タクティクス AI トレーニング（Embedding + MaxPooling版）

## 改善点
- **Embedding**: 数字ごとの「強さ」をAIが自動で学習
- **GlobalMaxPooling**: 手札枚数に関わらず同じ計算量・同じ出力サイズ
- **可変手札対応**: 5枚でも20枚でも同じモデルで処理

| カテゴリ | 操作 |
|---|---|
| 0 | パス |
| 1 | 約数強奪攻撃 |
| 2 | 足し算 |
| 3 | 引き算 |
| 4 | 商×余 |
| 5 | 桁和で分裂 |
| 6 | GCD掛け算 |

In [ ]:
!pip install tensorflowjs scikit-learn -q
print('完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random
from sklearn.model_selection import train_test_split

MAX_HAND    = 20   # 手札上限（これ以上は切り捨て）
EMBED_DIM   = 16   # 数字1枚あたりの特徴量の次元数
NUM_ACTIONS = 7    # 操作の種類
MAX_CARD    = 150  # カードの最大値

print(f'TensorFlow: {tf.__version__}')

## ゲームシミュレーター

In [ ]:
def gcd(a, b):
    while b: a, b = b, a % b
    return a

def digit_sum(n):
    return sum(int(d) for d in str(n))

class MathCardGame:
    def __init__(self, config=None):
        self.config = config or {
            'initHandCount': 5, 'initMaxNum': 10,
            'drawCount': 2, 'drawMaxNum': 20,
            'handLimitNum': MAX_CARD, 'winScore': 100, 'maxTurns': 10,
        }

    def reset(self):
        cfg = self.config
        n = cfg['initHandCount']
        self.hands = {
            'P1': [random.randint(1, cfg['initMaxNum']) for _ in range(n)],
            'P2': [random.randint(1, cfg['initMaxNum']) for _ in range(n)],
        }
        self.scores = {'P1': 0, 'P2': 0}
        self.turn_count = 1
        self.current_player = 'P1'
        self.done = False
        self.winner = None

    def opponent(self, pid): return 'P2' if pid == 'P1' else 'P1'

    def get_inputs(self, pid):
        """モデルへの3種類の入力を返す"""
        my   = self.hands[pid]
        enm  = self.hands[self.opponent(pid)]
        # 0パディング（0=空きスロット）
        my_pad  = (my[:MAX_HAND]  + [0]*MAX_HAND)[:MAX_HAND]
        enm_pad = (enm[:MAX_HAND] + [0]*MAX_HAND)[:MAX_HAND]
        scalars = [
            self.scores[pid] / 100.0,
            self.scores[self.opponent(pid)] / 100.0,
            self.turn_count / 10.0,
            len(my)  / MAX_HAND,   # 実際の手札枚数
            len(enm) / MAX_HAND,
        ]
        return my_pad, enm_pad, scalars

    # ─── 最善候補ヘルパー ───
    def best_attack(self, pid):
        enm = self.hands[self.opponent(pid)]
        best, bg = None, 0
        for i, a in enumerate(self.hands[pid]):
            if a == 1: continue
            g = sum(n for n in enm if n % a == 0)
            if g > bg: bg = g; best = (i, a, g)
        return best

    def best_add(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r = h[i]+h[j]
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def best_sub(self, pid):
        h = self.hands[pid]; best, bv = None, 0
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r = h[i]-h[j]
                if r>0 and r>bv: bv=r; best=(i,j,r)
        return best

    def best_divprod(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j or h[j]==0: continue
                r = (h[i]//h[j])*(h[i]%h[j])
                if r>0 and r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def best_dsd(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i, num in enumerate(h):
            ds = digit_sum(num)
            if ds==0: continue
            q, r = divmod(num, ds)
            if q<=lim and q>bv: bv=q; best=(i,q,r)
        return best

    def best_gm(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                g = gcd(h[i], h[j])
                if g<=1: continue
                r = h[i]*g
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def get_optimal_action(self, pid):
        atk = self.best_attack(pid)
        add = self.best_add(pid)
        sub = self.best_sub(pid)
        dp  = self.best_divprod(pid)
        dsd = self.best_dsd(pid)
        gm  = self.best_gm(pid)
        ms  = self.scores[pid]
        win = self.config['winScore']

        if atk and self.turn_count>1 and ms+atk[2]>=win: return 1
        if gm  and gm[2]>=30:  return 6
        if atk and self.turn_count>1 and atk[2]>=10: return 1
        if add and add[2]>=20:  return 2
        if dp  and dp[2]>=15:   return 4
        if atk and self.turn_count>1: return 1
        if gm  and gm[2]>=10:  return 6
        if dsd and dsd[1]>=5:  return 5
        if add: return 2
        if dp:  return 4
        if sub: return 3
        return 0

    def execute_action(self, pid, action):
        h = self.hands[pid]
        lim = self.config['handLimitNum']
        eid = self.opponent(pid)

        if action == 1:
            atk = self.best_attack(pid)
            if atk:
                i, v, gain = atk
                self.hands[eid] = [n for n in self.hands[eid] if n%v!=0]
                h.pop(i); self.scores[pid] += gain
                if self.scores[pid] >= self.config['winScore']:
                    self.done = True; self.winner = pid
        elif action == 2:
            a = self.best_add(pid)
            if a: i,j,r=a; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 3:
            s = self.best_sub(pid)
            if s: i,j,r=s; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 4:
            d = self.best_divprod(pid)
            if d: i,j,r=d; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 5:
            d = self.best_dsd(pid)
            if d: idx,q,r=d; h.pop(idx); h.append(q); (h.append(r) if r>0 else None)
        elif action == 6:
            g = self.best_gm(pid)
            if g: i,j,r=g; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)

        self.hands[pid] = h
        self.current_player = eid
        if self.current_player == 'P1':
            self.turn_count += 1
            if self.turn_count > self.config['maxTurns']:
                self.done = True
                self.winner = max(self.scores, key=lambda p: self.scores[p])

print('ゲームシミュレーター準備完了！')

## トレーニングデータ生成

In [ ]:
NUM_EPISODES = 30000

my_hands_data, enemy_hands_data, scalars_data, y_data = [], [], [], []
game = MathCardGame()

for ep in range(NUM_EPISODES):
    game.reset()
    while not game.done:
        pid = game.current_player
        my_h, enm_h, sc = game.get_inputs(pid)
        action = game.get_optimal_action(pid)
        my_hands_data.append(my_h)
        enemy_hands_data.append(enm_h)
        scalars_data.append(sc)
        y_data.append(action)
        game.execute_action(pid, action)
    if (ep+1) % 10000 == 0:
        print(f'{ep+1}/{NUM_EPISODES} 完了... データ数: {len(y_data)}')

my_hands_data    = np.array(my_hands_data,    dtype=np.int32)
enemy_hands_data = np.array(enemy_hands_data, dtype=np.int32)
scalars_data     = np.array(scalars_data,     dtype=np.float32)
y_data           = np.array(y_data,           dtype=np.int32)

print(f'\n総データ数: {len(y_data)}')
names = ['パス','攻撃','足し算','引き算','商×余','桁和分裂','GCD掛け']
u, c = np.unique(y_data, return_counts=True)
for i, cnt in zip(u, c):
    print(f'  {i}({names[i]}): {cnt}件 ({cnt/len(y_data)*100:.1f}%)')

## モデル構築（Embedding + GlobalMaxPooling）

```
my_hand [20個の整数]   → Embedding(16次元) → GlobalMaxPool → [16]
enemy_hand [20個の整数] → Embedding(16次元) → GlobalMaxPool → [16]  (重み共有)
scalars [5個の小数]    ─────────────────────────────────────→ [5]
                                                              ↓ 結合
                                                         Dense(128)
                                                         Dense(64)
                                                         Dense(7) → 操作確率
```

> **なぜMaxPoolが効率的か**
> - 手札5枚 → 5回だけEmbedding計算、残りは0（パディング）
> - 0のEmbeddingは実際のカードより小さい値になるよう学習されるため最大値に影響しない
> - 枚数が増えても後続のDense層の計算量は変わらない

In [ ]:
from tensorflow.keras.utils import to_categorical

# データ分割
idx = np.arange(len(y_data))
train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=42)

def make_dataset(indices, batch_size=256, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((
        {
            'my_hand':    my_hands_data[indices],
            'enemy_hand': enemy_hands_data[indices],
            'scalars':    scalars_data[indices],
        },
        to_categorical(y_data[indices], NUM_ACTIONS)
    ))
    if shuffle: ds = ds.shuffle(10000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_idx)
val_ds   = make_dataset(val_idx, shuffle=False)

# ─── モデル定義 ───
my_hand_input    = tf.keras.Input(shape=(MAX_HAND,), dtype='int32', name='my_hand')
enemy_hand_input = tf.keras.Input(shape=(MAX_HAND,), dtype='int32', name='enemy_hand')
scalar_input     = tf.keras.Input(shape=(5,),        dtype='float32', name='scalars')

# 自分と相手で同じEmbeddingを共有（数字の意味は同じ）
card_embedding = tf.keras.layers.Embedding(
    input_dim  = MAX_CARD + 1,  # 0(パディング) ～ 150
    output_dim = EMBED_DIM,
    mask_zero  = True,          # 0パディングを無視
    name       = 'card_emb'
)

my_emb    = card_embedding(my_hand_input)    # (batch, 20, 16)
enemy_emb = card_embedding(enemy_hand_input) # (batch, 20, 16)

# MaxPool: 手札の中で最も重要な特徴だけを取り出す
my_pool    = tf.keras.layers.GlobalMaxPooling1D(name='my_pool')(my_emb)    # (batch, 16)
enemy_pool = tf.keras.layers.GlobalMaxPooling1D(name='enemy_pool')(enemy_emb) # (batch, 16)

# 結合: 16 + 16 + 5 = 37次元
combined = tf.keras.layers.Concatenate(name='combined')([my_pool, enemy_pool, scalar_input])

x = tf.keras.layers.Dense(128, activation='relu')(combined)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
output = tf.keras.layers.Dense(NUM_ACTIONS, activation='softmax', name='action')(x)

model = tf.keras.Model(
    inputs  = [my_hand_input, enemy_hand_input, scalar_input],
    outputs = output
)

model.compile(
    optimizer = tf.keras.optimizers.Adam(0.001),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]

history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=60, callbacks=callbacks, verbose=1
)
print(f'\n最高検証精度: {max(history.history["val_accuracy"])*100:.1f}%')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['accuracy','loss'], ['精度','損失']):
    ax.plot(history.history[key], label='訓練')
    ax.plot(history.history[f'val_{key}'], label='検証')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

# 学習したEmbeddingを可視化（数字の強さマップ）
weights = model.get_layer('card_emb').get_weights()[0]  # (151, 16)
strength = np.linalg.norm(weights, axis=1)  # 各数字のベクトルの大きさ
plt.figure(figsize=(14, 3))
plt.bar(range(1, min(51, MAX_CARD+1)), strength[1:51])
plt.title('数字の強さ（AIが学習した埋め込みベクトルのノルム）1〜50')
plt.xlabel('数字'); plt.ylabel('強さ')
plt.show()

## TensorFlow.js 形式でエクスポート

In [ ]:
import os, shutil
OUT = 'tfjs_model'
if os.path.exists(OUT): shutil.rmtree(OUT)
tfjs.converters.save_keras_model(model, OUT)
print('エクスポート完了！')
for f in os.listdir(OUT):
    print(f'  {f} ({os.path.getsize(os.path.join(OUT,f))/1024:.1f} KB)')

shutil.make_archive('tfjs_model', 'zip', 'tfjs_model')
from google.colab import files
files.download('tfjs_model.zip')
print('ダウンロード開始！')

## 動作テスト

In [ ]:
names = ['パス','攻撃','足し算','引き算','商×余','桁和分裂','GCD掛け']
tests = [
    {'p1':[6,3,10,7,2], 'p2':[12,18,5,9,4], 'desc':'攻撃チャンスあり'},
    {'p1':[5,8,3,2,7],  'p2':[1,1,1,1,1],  'desc':'合成が有利'},
    {'p1':[6,9,4,3,2],  'p2':[8,10,6,4,3], 'desc':'GCD有効（6と9のGCD=3）'},
    {'p1':[2,3],        'p2':[4,6,8,10,12,14,16,18,20,22,24,26], 'desc':'手札少ない vs 相手12枚'},
]

for t in tests:
    g = MathCardGame()
    g.reset()
    g.hands['P1'] = t['p1']; g.hands['P2'] = t['p2']
    g.turn_count = 3; g.scores = {'P1': 20, 'P2': 30}

    my_h, enm_h, sc = g.get_inputs('P1')
    pred = model.predict(
        {'my_hand': np.array([my_h]), 'enemy_hand': np.array([enm_h]), 'scalars': np.array([sc])},
        verbose=0
    )[0]
    chosen  = int(np.argmax(pred))
    optimal = g.get_optimal_action('P1')

    print(f'【{t["desc"]}】')
    print(f'  自分({len(t["p1"])}枚): {t["p1"]}  相手({len(t["p2"])}枚): {t["p2"]}')
    print(f'  AI: {names[chosen]} | 最適: {names[optimal]} | {"✅" if chosen==optimal else "❌"}')
    print(f'  確率: {" ".join(f"{names[i]}:{pred[i]:.2f}" for i in range(NUM_ACTIONS))}')
    print()